# Crop Yield — Feature Exploration (companion notebook)

**Purpose.** One-off EDA on the data the experimentation notebook trains against. Answers questions like:
- How does yield distribute by state and by year?
- Which weather features correlate with yield, and does correlation vary by crop?
- Are there obvious data-quality issues (zeros, impossible values, wide gaps)?
- Is test-year weather weird vs train? (Distribution shift detection.)

Run this **before** iterating on models — it's the fastest way to spot data problems that would otherwise show up as mysteriously bad model scores. Re-use the Setup / Load cells from the main notebook; this notebook imports them via `%run` so we don't duplicate code.

## 1 · Setup

In [ ]:
%pip install -q pandas numpy matplotlib seaborn boto3 pyarrow fastparquet 2>&1 | tail -n 3

In [ ]:
import os, sys, time, tempfile, subprocess
from pathlib import Path
from datetime import date, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

REPO_URL = os.environ.get('ADS_REPO_URL', 'https://github.com/YOUR_ORG/Agricultural_Data_Analysis.git')
REPO_DIR = '/content/Agricultural_Data_Analysis' if os.path.isdir('/content') else './Agricultural_Data_Analysis'
if not os.path.isdir(REPO_DIR):
    if os.path.isfile('backend/models/yield_model.py'):
        REPO_DIR = os.getcwd()
    else:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110
pd.set_option('display.max_columns', 60)

# Cache used by both notebooks
CACHE = Path(tempfile.gettempdir()) / 'ads_yield_cache'
CACHE.mkdir(exist_ok=True)

In [ ]:
# 1.1 — S3 client + loaders (inline here for simplicity; keep in sync with the main notebook)
import boto3
from botocore import UNSIGNED
from botocore.client import Config

def make_s3_client(region='us-east-2'):
    if os.environ.get('AWS_ACCESS_KEY_ID'):
        return boto3.client('s3', region_name=region)
    return boto3.client('s3', region_name=region, config=Config(signature_version=UNSIGNED))
s3 = make_s3_client()

BUCKET = 'usda-analysis-datasets'
NASS_PREFIX = 'survey_datasets/partitioned_states_counties/'
WEATHER_KEY = 'weather/ghcn_processed/county_weather_2000_2025.parquet'

def sync_nass():
    dest = CACHE / 'county_parquets'
    dest.mkdir(exist_ok=True)
    if list(dest.glob('*.parquet')):
        return dest
    paginator = s3.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=BUCKET, Prefix=NASS_PREFIX):
        for obj in page.get('Contents', []):
            key = obj['Key']
            if key.endswith('/'):
                continue
            local = dest / Path(key).name
            if not local.exists():
                s3.download_file(BUCKET, key, str(local))
    return dest

def load_yields(crop_nass: str) -> pd.DataFrame:
    parquet_dir = sync_nass()
    rows = []
    for pq in sorted(parquet_dir.glob('*.parquet')):
        if pq.name.startswith('_'):
            continue
        df = pd.read_parquet(pq)
        if crop_nass == 'WHEAT':
            class_ok = df['class_desc'].isin(['ALL CLASSES', 'WINTER', 'SPRING, (EXCL DURUM)'])
        else:
            class_ok = df['class_desc'] == 'ALL CLASSES'
        sub = df[(df['agg_level_desc']=='COUNTY') & (df['statisticcat_desc']=='YIELD') & (df['commodity_desc']==crop_nass)
                 & class_ok & (df['unit_desc'].str.contains('BU / ACRE', case=False, na=False))
                 & (df['domain_desc']=='TOTAL') & (df['freq_desc']=='ANNUAL')]
        if sub.empty:
            continue
        sub = sub.copy()
        sub['fips'] = sub['state_fips_code'].astype(str).str.zfill(2) + sub['county_code'].astype(str).str.zfill(3)
        for _, r in sub.iterrows():
            try:
                val = float(str(r.get('value_num', r.get('Value', ''))).replace(',', ''))
            except (ValueError, TypeError):
                continue
            if val <= 0 or np.isnan(val):
                continue
            rows.append({'fips': r['fips'], 'year': int(r['year']), 'yield_bu': val, 'state_fips': r['fips'][:2]})
    return pd.DataFrame(rows).drop_duplicates(subset=['fips', 'year'], keep='last').reset_index(drop=True)

def load_weather():
    local = CACHE / 'county_weather.parquet'
    if not local.exists():
        try:
            s3.download_file(BUCKET, WEATHER_KEY, str(local))
        except Exception as exc:
            print(f'weather download failed: {exc}')
            return pd.DataFrame()
    return pd.read_parquet(local)

## 2 · Yield distributions

**Looking for:** bimodality (dry-land vs irrigated), long left tails (crop failures worth investigating), year-over-year shifts that could be test-set shocks.

In [ ]:
CROP = 'corn'  # change to 'soybean' or 'wheat' (NASS names: CORN / SOYBEANS / WHEAT)
CROP_NASS = {'corn': 'CORN', 'soybean': 'SOYBEANS', 'wheat': 'WHEAT'}[CROP]
yields = load_yields(CROP_NASS)
print(f'{len(yields):,} rows, {yields["fips"].nunique():,} counties, years {yields["year"].min()}-{yields["year"].max()}')
yields.describe().T

In [ ]:
# 2.1 — Overall yield distribution + by decade
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(yields['yield_bu'], bins=60, color='#22c55e', edgecolor='white', alpha=0.85)
axes[0].set_title(f'{CROP_NASS} yield distribution (all years, all counties)')
axes[0].set_xlabel('bu/acre'); axes[0].set_ylabel('count')

yields['decade'] = (yields['year'] // 10) * 10
for dec, grp in yields.groupby('decade'):
    axes[1].hist(grp['yield_bu'], bins=40, alpha=0.4, label=f'{dec}s', density=True)
axes[1].legend(title='Decade')
axes[1].set_title('Distribution shift over decades (normalized)')
axes[1].set_xlabel('bu/acre'); axes[1].set_ylabel('density')
plt.tight_layout(); plt.show()

In [ ]:
# 2.2 — Year-over-year median yield. Look for the step-change that training will need to handle.
annual = yields.groupby('year')['yield_bu'].agg(['median', 'mean', 'std', 'count']).reset_index()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(annual['year'], annual['median'], marker='o', color='#22c55e', label='median')
ax.fill_between(annual['year'], annual['median']-annual['std'], annual['median']+annual['std'], alpha=0.15, color='#22c55e', label='±1σ')
ax.axvspan(2020, 2022, alpha=0.08, color='#f59e0b', label='val years')
ax.axvspan(2023, 2024, alpha=0.12, color='#ef4444', label='test years')
ax.set_title(f'{CROP_NASS} median county yield over time')
ax.set_xlabel('year'); ax.set_ylabel('median bu/acre'); ax.legend()
plt.tight_layout(); plt.show()
display(annual.tail(10))

In [ ]:
# 2.3 — State-level ranking (top 20 producing states by median yield)
state_med = yields.groupby('state_fips')['yield_bu'].agg(['median', 'count']).reset_index()
state_med = state_med[state_med['count'] >= 50].sort_values('median', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(state_med['state_fips'], state_med['median'], color='#3b82f6', alpha=0.8)
ax.invert_yaxis()
ax.set_title(f'{CROP_NASS} — top 20 states by median yield (counties × years, ≥50 obs)')
ax.set_xlabel('median bu/acre'); ax.set_ylabel('state FIPS')
plt.tight_layout(); plt.show()

## 3 · Weather features vs yield

**Looking for:** which raw weather signals have a monotonic or u-shaped relationship with yield, hint at feature transforms worth adding, and whether the 2023–2024 test years look anomalous against 2000–2019 train.

In [ ]:
# 3.1 — Build per (fips, year) summer-season weather summary (Jun-Aug)
weather = load_weather()
print(f'{len(weather):,} weather-day rows')

weather['date_parsed'] = pd.to_datetime(weather['date'], format='%Y-%m-%d', errors='coerce')
weather['year'] = weather['date_parsed'].dt.year
weather['month'] = weather['date_parsed'].dt.month
summer = weather[(weather['month'] >= 6) & (weather['month'] <= 8)].copy()

agg = (
    summer.groupby(['fips', 'year'])
    .agg(tmax_jja_mean=('tmax_f', 'mean'),
         tmin_jja_mean=('tmin_f', 'mean'),
         prcp_jja_total=('prcp_in', 'sum'),
         hot_days_jja=('tmax_f', lambda s: (s > 95).sum()))
    .reset_index()
)
wy = yields.merge(agg, on=['fips', 'year'], how='inner')
print(f'Joined: {len(wy):,} yield-year × weather rows')

In [ ]:
# 3.2 — Correlation heatmap
corr_cols = ['yield_bu', 'tmax_jja_mean', 'tmin_jja_mean', 'prcp_jja_total', 'hot_days_jja', 'year']
corr = wy[corr_cols].corr(method='spearman')

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title(f'Spearman correlation — {CROP_NASS} yield vs JJA weather')
plt.tight_layout(); plt.show()

In [ ]:
# 3.3 — Bivariate relationships. Helpful for spotting u-shaped effects linear models miss.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sample = wy.sample(min(5000, len(wy)), random_state=0)
for ax, xcol, xlab in zip(
    axes,
    ['tmax_jja_mean', 'prcp_jja_total', 'hot_days_jja'],
    ['Mean daily TMAX (Jun-Aug, °F)', 'Total precipitation (Jun-Aug, in)', 'Hot days TMAX>95°F (Jun-Aug)'],
):
    ax.scatter(sample[xcol], sample['yield_bu'], alpha=0.1, s=6, color='#1e293b')
    # LOWESS smooth
    try:
        from statsmodels.nonparametric.smoothers_lowess import lowess
        s = lowess(sample['yield_bu'], sample[xcol], frac=0.2, it=0, return_sorted=True)
        ax.plot(s[:, 0], s[:, 1], color='#f97316', lw=2.5, label='LOWESS')
    except Exception:
        pass
    ax.set_xlabel(xlab); ax.set_ylabel('yield (bu/acre)')
plt.suptitle(f'{CROP_NASS} yield vs summer weather', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# 3.4 — Distribution shift: do the test years (2023-24) look anomalous compared to training years?
split = wy['year'].apply(lambda y: 'train ≤2019' if y <= 2019 else ('val 2020-22' if y <= 2022 else 'test 2023-24'))
wy2 = wy.assign(_split=split)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, title in zip(
    axes,
    ['tmax_jja_mean', 'prcp_jja_total', 'hot_days_jja'],
    ['Mean daily TMAX (JJA)', 'Total precipitation (JJA)', 'Hot days (JJA)'],
):
    for split_name, color in [('train ≤2019', '#3b82f6'), ('val 2020-22', '#f59e0b'), ('test 2023-24', '#ef4444')]:
        data = wy2.loc[wy2['_split'] == split_name, col].dropna()
        ax.hist(data, bins=40, alpha=0.5, label=f'{split_name} (n={len(data):,})', density=True, color=color)
    ax.set_title(title); ax.legend(fontsize=8)
plt.suptitle(f'Distribution shift check — {CROP_NASS} ({len(wy2):,} rows)', y=1.02)
plt.tight_layout(); plt.show()

# If the test distribution is sharply offset from train, pure ML won't save you —
# the model is being asked to extrapolate. Consider weighting recent years
# higher, or using a conformal prediction approach that reports its own uncertainty.

## 4 · Data quality checks

Quick sanity checks that catch the most common silent data bugs.

In [ ]:
print('=== Yield quality ===')
print(f'  Duplicate (fips, year):     {yields.duplicated(subset=["fips","year"]).sum()}')
print(f'  Yields <= 0:                {(yields["yield_bu"] <= 0).sum()}')
print(f'  Yields > 400 (implausible): {(yields["yield_bu"] > 400).sum()}')
print(f'  NaN yields:                 {yields["yield_bu"].isna().sum()}')

# Counties with sparse history — these will be dropped by the feature builder anyway
sparse = yields.groupby('fips').size()
print(f'  Counties with <5 years:     {(sparse < 5).sum()} of {sparse.shape[0]}')

print('\n=== Weather quality ===')
if not weather.empty:
    print(f'  NaN TMAX:    {weather["tmax_f"].isna().mean():.2%}')
    print(f'  NaN TMIN:    {weather["tmin_f"].isna().mean():.2%}')
    print(f'  NaN PRCP:    {weather["prcp_in"].isna().mean():.2%}')
    print(f'  TMAX < TMIN: {((weather["tmax_f"] < weather["tmin_f"]) & weather["tmax_f"].notna() & weather["tmin_f"].notna()).sum()}')
    print(f'  Impossible TMAX (>130°F): {(weather["tmax_f"] > 130).sum()}')
    print(f'  Impossible PRCP (>20in): {(weather["prcp_in"] > 20).sum()}')
else:
    print('  (weather not loaded)')

## 5 · Feature hypotheses to try

Things the EDA above suggests might help (based on typical corn/soy patterns — verify for your own run):

| Hypothesis | How to add |
|---|---|
| **VPD stress** — high TMAX alone undersells heat stress when it's also dry | Add a VPD proxy: `tmax * (1 - prcp_deficit/prcp_normal)` |
| **Nighttime warming** — TMIN rising faster than TMAX hurts grain fill | Add `tmin_jja_mean` as a standalone feature |
| **Early-season drought** — May moisture sets root depth | Extend weather window to include May precipitation |
| **Accumulated hot-day penalty** — sigmoid of `hot_days` beats linear | Engineer `1 / (1 + exp(-(hot_days-10)/3))` |
| **State-level yield anomaly** — relative yield vs state average carries signal | `yield / state_year_mean` as a target transform |

Take these to the main experimentation notebook and compare via MLflow.

---

*Exploration done. Go iterate.*